In [ ]:
!mkdir -p /root/.config/kaggle
!cp /content/kaggle.json /root/.config/kaggle/
!chmod 600 /root/.config/kaggle/kaggle.json

In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()
api.dataset_download_files("olistbr/brazilian-ecommerce", path="data/", unzip=True)

df_pay   = pd.read_csv("data/olist_order_payments_dataset.csv")
df_orders = pd.read_csv("data/olist_orders_dataset.csv")

df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'])
# Объединяем по order_id
df = df_pay.merge(df_orders[['order_id', 'order_purchase_timestamp']],
                  on="order_id", how="inner")
print(df.shape)

Dataset URL: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
(103886, 6)


In [ ]:
# Выберем две контрольные даты для сравнения. Например:
#   - control_date = "2018-04-25"
#   - new_date     = "2018-04-26"

control_date = pd.to_datetime("2018-04-25")
new_date     = pd.to_datetime("2018-04-26")

# Фильтруем оплаты, совершённые именно в контрольную дату (с 00:00 до 23:59:59)
df_control = df[
    (df['order_purchase_timestamp'] >= control_date) &
    (df['order_purchase_timestamp'] <  control_date + pd.Timedelta(days=1))
].copy()

# Фильтруем оплаты, совершённые в новую дату
df_new = df[
    (df['order_purchase_timestamp'] >= new_date) &
    (df['order_purchase_timestamp'] <  new_date + pd.Timedelta(days=1))
].copy()

print(f"Число записей в control: {len(df_control)}")
print(f"Число записей в new: {len(df_new)}")

Число записей в control: 290
Число записей в new: 265


In [ ]:
def psi(ref_array: np.ndarray,
        tgt_array: np.ndarray,
        bins: int = 10,
        fixed: bool = True,
        eps: float = 1e-8) -> float:
    """
    Рассчет PSI между двумя массивами числовых значений.

    Args:
        ref_array (np.ndarray): Reference (control) array.
        tgt_array (np.ndarray): Target (new) array.
        bins (int): Number of bins.
        fixed (bool): Если True, бины по равномерному размаху.
                      Если False, бины по квантилям ref_array.
        eps (float): Малое значение, чтобы избежать деления на ноль.

    Returns:
        float: Значение PSI.
    """
    # Строим бины
    if fixed:
        # Минимум и максимум по объединённым значениям
        overall_min = min(ref_array.min(), tgt_array.min())
        overall_max = max(ref_array.max(), tgt_array.max())
        # Разбиваем на равномерные интервалы
        breaks = np.linspace(overall_min, overall_max, bins + 1)
    else:
        # Бины на основе квантилей ref_array
        breaks = np.quantile(ref_array, np.linspace(0, 1, bins + 1))

    # Считаем частоты по бинам (normed)
    ref_counts, _ = np.histogram(ref_array, bins=breaks)
    tgt_counts, _ = np.histogram(tgt_array, bins=breaks)

    # Переводим в доли
    ref_perc = ref_counts / len(ref_array)
    tgt_perc = tgt_counts / len(tgt_array)

    # Избегаем нулей
    ref_perc = np.where(ref_perc == 0, eps, ref_perc)
    tgt_perc = np.where(tgt_perc == 0, eps, tgt_perc)

    # Формула PSI
    psi_values = (ref_perc - tgt_perc) * np.log(ref_perc / tgt_perc)
    return np.sum(psi_values)

In [ ]:
# Подготовим массивы значений оплаты за контрольную и новую дату.
ref_payments = df_control['payment_value'].values
new_payments = df_new['payment_value'].values

print(f"Средний payment_value в контрольной группе: {ref_payments.mean():.4f}")
print(f"Средний payment_value в новой группе:       {new_payments.mean():.4f}")

# PSI без аномалий
psi_value = psi(ref_array=ref_payments, tgt_array=new_payments, bins=10, fixed=True)
print(f"PSI между {control_date.date()} и {new_date.date()} (без аномалий): {psi_value:.4f}")

Средний payment_value в контрольной группе: 163.2982
Средний payment_value в новой группе:       139.2935
PSI между 2018-04-25 и 2018-04-26 (без аномалий): 0.1235


In [ ]:
# Добавим несколько «экстремальных» значений в новую выборку:
#   - Очень большие оплаты (например, 1000, 1500, 2000 → редкие, но излишне крупные транзакции)
#   - Очень маленькие (например, 0.01, 0.05 → мнимые «пустые» или явно ошибочные)
#   - Можно варьировать количество таких значений в зависимости от желаемого эффекта
anom_values = [0.01, 0.05, 1000, 1500, 2000]
new_payments_anom = np.concatenate([new_payments, np.array(anom_values * 10)])
print(f"Средний payment_value в контрольной группе: {ref_payments.mean():.4f}")
print(f"Средний payment_value в новой группе (с аномалиями): {new_payments_anom.mean():.4f}")

# Пересчитаем PSI
psi_value_anom = psi(ref_array=ref_payments,
                     tgt_array=new_payments_anom,
                     bins=10,
                     fixed=True)

print(f"PSI между {control_date.date()} и {new_date.date()} (с аномалиями): {psi_value_anom:.4f}")

Средний payment_value в контрольной группе: 163.2982
Средний payment_value в новой группе (с аномалиями): 260.0424
PSI между 2018-04-25 и 2018-04-26 (с аномалиями): 0.6678
